# Let's go PRO!

Advanced RAG Techniques!

Let's start by digging into ingest:

1. No LangChain! Just native for maximum flexibility
2. Let's use an LLM to divide up chunks in a sensible way
3. Let's use the best chunk size and encoder from yesterday
4. Let's also have the LLM rewrite chunks in a way that's most useful ("document pre-processing")

In [21]:
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from ollama import Client


load_dotenv(override=True)

MODEL = "ollama/llama3.1:8b" # "ollama/llama3.2"
DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "Qwen/Qwen3-Embedding-0.6B"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500
#ollama_url = "http://localhost:11434/v1"
#openai = OpenAI(api_key="ollama", base_url=ollama_url)
ollama_url = "http://localhost:11434" 
ollama_client = Client(host=ollama_url)
#import os
#from openai import OpenAI
#import torch
#from transformers import AutoTokenizer, AutoModel
#from huggingface_hub import login, InferenceClient
#hf_token = os.environ.get("HF_TOKEN")
#if hf_token:
#    print("Logging in...")
#    login(hf_token, add_to_git_credential=True)
#client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=hf_token)
# Initialize the native Hugging Face client
#hf_client = InferenceClient(api_key=hf_token)


In [22]:
# Inspired by LangChain's Document - let's have something similar

class Result(BaseModel):
    page_content: str
    metadata: dict

In [23]:
# A class to perfectly represent a chunk

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]

## Three steps:

1. Fetch documents from the knowledge base, like LangChain did
2. Call an LLM to turn documents into Chunks
3. Store the Chunks in Chroma

That's it!

### Let's start with Step 1

In [24]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        print(f"Load {doc_type}...")
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [5]:
documents = fetch_documents()

Load products...
Load contracts...
Load company...
Load employees...
Loaded 76 documents


### Donezo! On to Step 2 - make the chunks

In [25]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [9]:
print(make_prompt(documents[0]))


You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: products
The document has been retrieved from: knowledge-base/products/Rellm.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into 8 chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

# Product Summary

# Rellm: AI-Powered Enterprise 

In [26]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [11]:
make_messages(documents[0])

[{'role': 'user',
  'content': "\nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\nThe document is from the shared drive of a company called Insurellm.\nThe document is of type: products\nThe document has been retrieved from: knowledge-base/products/Rellm.md\n\nA chatbot will use these chunks to answer questions about the company.\nYou should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.\nThis document should probably be split into 8 chunks, but you can have more or less as appropriate.\nThere should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\nFor each chunk, you should provide a headline, a summary, and the original text of the chunk.\nTogether your chunks should represent the entire document with overlap.\n\nHere is the document:\n\n#

In [27]:
def process_document(document):
    messages = make_messages(document)
    response = completion(model="ollama/llama3.1:8b", messages=messages, response_format=Chunks, base_url="http://localhost:11434")
    reply = response.choices[0].message.content
    try:
        doc_as_chunks = Chunks.model_validate_json(reply).chunks
    except Exception as e:  
        print(f"An exception: {e}")  
        response = completion(model=MODEL, messages=messages, response_format=Chunks, base_url="http://localhost:11434")
        reply = response.choices[0].message.content
        doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [15]:
process_document(documents[50])

[Result(page_content='# HR Record\n\n\n\n# HR Record\n# Amanda Foster', metadata={'source': 'knowledge-base/employees/Amanda Foster.md', 'type': 'employees'}),
 Result(page_content='# Amanda Foster\n\n\n\n# Amanda Foster\n## Summary\n- **Date of Birth:** August 14, 1982\n- **Job Title:** HR Business Partner\n- **Location:** San Francisco, California\n- **Current Salary:** $115,000', metadata={'source': 'knowledge-base/employees/Amanda Foster.md', 'type': 'employees'}),
 Result(page_content='HR Record - Summary\n\n\n\n## Summary\n- **Date of Birth:** August 14, 1982\n- **Job Title:** HR Business Partner\n- **Location:** San Francisco, California\n- **Current Salary:** $115,000', metadata={'source': 'knowledge-base/employees/Amanda Foster.md', 'type': 'employees'}),
 Result(page_content='Insurellm Career Progression\n\n\n\n## Insurellm Career Progression\n- **September 2016 - Present:** HR Business Partner\n  - Partners with engineering and product leadership teams\n  - Supports 85 emplo

In [28]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [17]:
chunks = create_chunks(documents)

100%|██████████| 76/76 [1:19:07<00:00, 62.47s/it]


In [18]:
print(len(chunks))

902


### Well that was easy! If a bit slow.

In the python module version, I sneakily use the multi-processing Pool to run this in parallel,
but if you get a Rate Limit Error you can turn this off in the code.

### Finally, Step 3 - save the embeddings

In [29]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)    
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    print("Generating vectors...")
    vectors = []
    for text in texts:
        response = ollama_client.embeddings(model="qwen3-embedding:0.6b", prompt=text)
        vectors.append(response['embedding'])

    collection = chroma.get_or_create_collection(collection_name)
    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]
    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")


In [20]:
create_embeddings(chunks)

Generating vectors...
Vectorstore created with 902 documents


# Nothing more to do here... right?

Wait! Didja think I'd forget??

In [30]:
# Connect to DB
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)


In [31]:
result = collection.get(include=['embeddings', 'documents', 'metadatas'])

vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [32]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [33]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

## And now - let's build an Advanced RAG!

We will use these techniques:

1. Reranking - reorder the rank results
2. Query re-writing

In [34]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [35]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order[:10])
    return [chunks[i - 1] for i in order[:10]]

In [36]:
RETRIEVAL_K = 10

def fetch_context_unranked(question):
    #query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    query = ollama_client.embeddings(model="qwen3-embedding:0.6b", prompt=question)
    results = collection.query(query_embeddings=query['embedding'], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [37]:
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)

In [75]:
for chunk in chunks:
    print(chunk.page_content[:15]+"...")

# Annual Perfor...
# Product Summa...
# Product Summa...
# Product Summa...
# Product Summa...
# Product Summa...
# Annual Perfor...
# Product Summa...
# Product Summa...
# HR Record # M...


In [39]:
reranked = rerank(question, chunks)

[8, 3, 5, 9, 7, 4, 1, 2, 6, 10]


In [40]:
for chunk in reranked:
    print(chunk.page_content[:30]+"...")

# HR Record

Employee career p...
# HR Record

Employee performa...
Annual Performance History

Pr...
Annual Performance History

Da...
**Advanced AI Claims Adjudicat...
IoT and Innovation Support

Th...
- **2023:** Rating: 4.9/5



-...
# Annual Performance History

...
# Other HR Notes

- **Recognit...
Feature 6: AI-Powered Litigati...


In [41]:
question = "Who went to Manchester University?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)

0


In [42]:
reranked = rerank(question, chunks)

[8, 1, 4, 2, 3, 12, 13, 6, 16, 5]


In [43]:
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)

1


In [45]:
reranked[1].page_content

"# Other HR Notes\n\nOther notes about Jessica Liu from Insurellm's HR.\n\n## Other HR Notes\n- **Education:** BS in Computer Science from University of Manchester"

In [27]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [28]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [29]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [70]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Your responde shuld be:
* Only correct grammar if necessary, never change the meaning. of the current question.
* Summarize the question if necessary.
* It shuld be a VERY short specific question. 
* Never mention the company name unless it's already present on the user's current questions.
* IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [71]:
rewrite_query("Who won the IIOTY award? and when", [])

'Who won the IIOTY award and when?'

In [ ]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

In [78]:
answer_question("Who won the IIOTY award? and when?", [])

What were the past winners of the IIOTY award?
[16, 20, 10, 2, 19, 18, 4, 1, 13, 15]


('According to our knowledge base, Maxine Thompson was recognized as the Insurellm Innovator of the Year (IIOTY) in 2023, receiving the prestigious IIOTY 2023 award.',
 [Result(page_content='Annual Performance History\n\nAnnual performance ratings and achievements for Alex Thomson.\n\n## Annual Performance History\n- **2022** - Rated as "Exceeds Expectations." Alex Thomson achieved 150% of the sales target within the first three months.\n- **2023** - Rated "Outstanding." Recognized for innovative lead-generation tactics which contributed to a 30% increase in qualified leads for the sales team.', metadata={'source': 'knowledge-base/employees/Alex Thomson.md', 'type': 'employees'}),
  Result(page_content='# HR Record # Maxine Thompson ## Annual Performance History\n\nFinal part of annual performance history for Maxine.\n\n\n- **2022**: *Outstanding*\n  Maxine continued her upward trajectory, successfully implementing machine learning algorithms to predict customer behavior, which was wel

In [79]:
answer_question("Who went to Manchester University?", [])

What Insurellm staff members have attended Manchester University?
[12, 2, 17, 18, 15, 20]


('According to our knowledge base, Jessica Liu has a BS in Computer Science from the University of Manchester.',
 [Result(page_content="## Client Portfolio Breakdown\n\nBreakdown of company's client portfolio.\n\n\n\n## Scale & Impact\n\nDespite its compact team size, Insurellm has built an impressive client portfolio with 32 active contracts across all product lines,\n\n### Marketplace & Infrastructure\n- **Markellm** - Marketplace connecting consumers with insurance providers (original flagship product)\n- **Claimllm** - AI-powered claims processing platform across all insurance lines\n\n\n\n## Client Portfolio Breakdown\n\nInsurellm's 32 active contracts span the full spectrum of insurance technology", metadata={'source': 'knowledge-base/company/overview.md', 'type': 'company'}),
  Result(page_content="# HR Record # Jessica Liu ## Compensation History # 2020\n\nJessica Liu's compensation history within Insurellm.\n\n\n## Other HR Notes\n- **Education:** BS in Computer Science from U